# AlphaKraken REST API Demo

Demonstrating how to retrieve data from the REST API.

In [ ]:
import pandas as pd
import requests

BASE_URL = "http://localhost:8090"
TIMEOUT = 10

In [ ]:
def get(  # noqa: PLR0913
    *,
    instrument_id: str | None = None,
    name_contains: str | None = None,
    project_id: str | None = None,
    date_from: str | None = None,
    date_to: str | None = None,
    limit: int | None = None,
    offset: int | None = None,
) -> dict:
    """Get data from the REST API."""
    params = {k: v for k, v in locals().items() if v is not None}
    return requests.get(f"{BASE_URL}/raw_files/", params=params, timeout=TIMEOUT).json()

In [ ]:
def to_dataframe(data: dict) -> pd.DataFrame:
    """Convert API response to a DataFrame, expanding metrics into columns prefixed by type."""
    rows = []
    for raw_file in data["data"]:
        row = {k: v for k, v in raw_file.items() if k != "metrics"}
        for m in raw_file.get("metrics", []):
            prefix = m.get("type", "metrics")
            for k, v in m.items():
                if k != "type":
                    row[f"{prefix}:{k}"] = v
        rows.append(row)
    return pd.DataFrame(rows)

## Get recent raw files

In [ ]:
data = get(limit=5)
print(f"Total: {data['total']}, returned: {len(data['data'])}")
data["data"][0]

## Filter by instrument and date range

In [ ]:
get(instrument_id="test1", date_from="2025-01-01", date_to="2025-12-31", limit=3)

## Search by name substring

In [ ]:
get(name_contains="hela", limit=3)

## Pagination

In [ ]:
page1 = get(limit=2, offset=0)
page2 = get(limit=2, offset=2)

print(f"Total: {page1['total']}")
print(f"Page 1 IDs: {[r['id'] for r in page1['data']]}")
print(f"Page 2 IDs: {[r['id'] for r in page2['data']]}")

## Load into DataFrame

In [ ]:
df = to_dataframe(get(limit=50, instrument_id="stellar1"))
df.head()